In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import cv2
import os
import numpy as np

base_path = '/content/drive/MyDrive/Handwriting'
csv_file_path = os.path.join(base_path, 'english.csv')

# Check if the CSV file exists
if not os.path.exists(csv_file_path):
    raise FileNotFoundError(f"The required file '{csv_file_path}' does not exist. Please ensure 'english.csv' is in the '/content/drive/MyDrive/Handwriting' directory of your Google Drive.")

df = pd.read_csv(csv_file_path)

target_size = (64, 64)
processed_data = []

print("Starting processing...")

for index, row in df.iterrows():
    full_img_path = os.path.join(base_path, str(row['image']))

    img = cv2.imread(full_img_path, cv2.IMREAD_GRAYSCALE)

    if img is not None:
        # A. Resize
        resized = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)

        # B. Normalize (Scale 0-255 to 0.0-1.0)

        normalized = resized.astype('float32') / 255.0

        # C. Flatten (64x64 matrix -> 4096 vector)
        flattened = normalized.flatten()

        processed_data.append(flattened)


X = np.array(processed_data)
y = df['label'].values

print(f"Final shape of X: {X.shape}")


FileNotFoundError: The required file '/content/drive/MyDrive/Handwriting/english.csv' does not exist. Please ensure 'english.csv' is in the '/content/drive/MyDrive/Handwriting' directory of your Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


In [ ]:
import os

base_path = '/content/drive/MyDrive/Handwriting'
csv_file_path = os.path.join(base_path, 'english.csv')

if os.path.exists(csv_file_path):
    print(f"The file '{csv_file_path}' exists and is accessible.")
else:
    print(f"Error: The file '{csv_file_path}' still does not exist or is not accessible. Please check your Google Drive path and file name.")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# 1. Base Perceptron Class
class Perceptron:
    def __init__(self, lr=0.01, epochs=50):
        self.lr = lr
        self.epochs = epochs
        self.weights = None
        self.bias = 0

    def step_function(self, x):
        return 1 if x >= 0 else 0

    def fit(self, X, y):
        self.weights = np.zeros(X.shape[1])
        self.bias = 0
        for _ in range(self.epochs):
            for x_i, target in zip(X, y):
                # Perceptron Weight Update Rule
                prediction = self.step_function(np.dot(x_i, self.weights) + self.bias)
                update = self.lr * (target - prediction)
                self.weights += update * x_i
                self.bias += update

    def get_score(self, X):
        return np.dot(X, self.weights) + self.bias

# 2. One-vs-Rest Wrapper
class OneVsRestPLA:
    def __init__(self, epochs=50):
        self.epochs = epochs
        self.classifiers = []
        self.classes = None

    def fit(self, X, y):
        self.classes = np.unique(y)
        for cls in self.classes:
            # Binary label: 1 if it's the current class, 0 otherwise
            y_binary = np.where(y == cls, 1, 0)
            model = Perceptron(epochs=self.epochs)
            model.fit(X, y_binary)
            self.classifiers.append(model)
            print(f"Trained class: {cls}")

    def predict(self, X):

        scores = np.array([clf.get_score(X) for clf in self.classifiers]).T
        class_indices = np.argmax(scores, axis=1)
        return self.classes[class_indices]

# 3. Accuracy Calculation
def get_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred) * 100

indices = np.arange(len(X))
np.random.shuffle(indices)
X_shuffled, y_shuffled = X[indices], y[indices]

# Train the model
ovr_pla = OneVsRestPLA(epochs=20)
ovr_pla.fit(X_shuffled, y_shuffled)

# Predict and Calculate Accuracy
y_pred = ovr_pla.predict(X_shuffled)
accuracy = get_accuracy(y_shuffled, y_pred)

print(f"\nFinal Accuracy: {accuracy:.2f}%")


In [ ]:
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
import numpy as np

le = LabelEncoder()
y = y.ravel()        # convert pandas column to array
y_encoded = le.fit_transform(y).astype(int)

num_classes = len(np.unique(y_encoded))
y_onehot = tf.one_hot(y_encoded, depth=num_classes)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y_onehot.numpy(), test_size=0.2, random_state=42)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

model = Sequential()

# Hidden Layer (ReLU)
model.add(Dense(128, input_shape=(4096,), activation='relu'))

# Output Layer (Softmax for Multiclass)
model.add(Dense(num_classes, activation='softmax'))

# Compile Model (Adam Optimizer)
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()


In [ ]:
history = model.fit(X_train, y_train,
                    epochs=20,
                    batch_size=32,
                    validation_data=(X_test, y_test))


In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy * 100)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam, SGD

def build_mlp(hidden_layers, activation, optimizer, lr, input_size, num_classes):
    model = Sequential()

    # First hidden layer
    model.add(Dense(hidden_layers[0], input_shape=(input_size,), activation=activation))

    # Additional hidden layers
    for neurons in hidden_layers[1:]:
        model.add(Dense(neurons, activation=activation))

    # Output layer
    model.add(Dense(num_classes, activation='softmax'))

    # Optimizer selection
    if optimizer == "adam":
        opt = Adam(learning_rate=lr)
    else:
        opt = SGD(learning_rate=lr)

    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
pla_accuracy = 21.70

from sklearn.metrics import precision_score, recall_score, f1_score

# Calculate precision, recall, and F1-score
# Use average='weighted' for multiclass classification to account for class imbalance
pla_precision = precision_score(y_shuffled, y_pred, average='weighted', zero_division=0)
pla_recall = recall_score(y_shuffled, y_pred, average='weighted', zero_division=0)
pla_f1_score = f1_score(y_shuffled, y_pred, average='weighted', zero_division=0)

print(f"PLA Model Accuracy: {pla_accuracy:.2f}%")
print(f"PLA Model Precision: {pla_precision:.2f}")
print(f"PLA Model Recall: {pla_recall:.2f}")
print(f"PLA Model F1-score: {pla_f1_score:.2f}")

In [ ]:
import numpy as np

# MLP Accuracy from previous output
mlp_accuracy = accuracy * 100

# Get predictions from MLP model
y_pred_mlp_raw = model.predict(X_test)

# Convert one-hot encoded y_test to single class labels
y_test_labels = np.argmax(y_test, axis=1)

# Convert MLP predicted probabilities to single class labels
y_pred_mlp_labels = np.argmax(y_pred_mlp_raw, axis=1)

# Calculate precision, recall, and F1-score for MLP model
mlp_precision = precision_score(y_test_labels, y_pred_mlp_labels, average='weighted', zero_division=0)
mlp_recall = recall_score(y_test_labels, y_pred_mlp_labels, average='weighted', zero_division=0)
mlp_f1_score = f1_score(y_test_labels, y_pred_mlp_labels, average='weighted', zero_division=0)

print(f"MLP Model Accuracy: {mlp_accuracy:.2f}%")
print(f"MLP Model Precision: {mlp_precision:.2f}")
print(f"MLP Model Recall: {mlp_recall:.2f}")
print(f"MLP Model F1-score: {mlp_f1_score:.2f}")

In [ ]:
import pandas as pd

# Create a dictionary to hold the metrics for each model
metrics_data = {
    'Model': ['PLA', 'MLP'],
    'Accuracy (%)': [pla_accuracy, mlp_accuracy],
    'Precision': [pla_precision, mlp_precision],
    'Recall': [pla_recall, mlp_recall],
    'F1-score': [pla_f1_score, mlp_f1_score]
}

# Create a DataFrame from the dictionary
metrics_df = pd.DataFrame(metrics_data)

# Display the DataFrame
print("\n--- Model Performance Comparison ---")
print(metrics_df.to_markdown(index=False, floatfmt=".2f"))

In [ ]:
#Confusion Matrix

from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_pred = model.predict(X_test)
cm = confusion_matrix(y_test,y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix = cm)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()